# Conditional Inference Results

This notebook inspects the two u128 conditional inference runs:

1. **Discrete class-conditional field model**: `nf_class_conditional_u128`, conditioned on field label only.
2. **Continuous conditional cosmology model**: `nf_conditional_u128`, conditioned on six CAMELS parameters.

It is designed to run from the repo root on Great Lakes after sampling jobs finish. Missing files are reported rather than treated as hard failures. The sample labels are explicit so the notebook does not accidentally switch between raw DDPM-style samples and any DPM/DDPM comparison files in the same directory.


In [ ]:
from __future__ import annotations

import json
import math
import os
import sys
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_CANDIDATES = [
    Path.cwd(),
    Path('/home/jiamingp/diffusion_models_repo'),
    Path('/Users/apple/AI/Diffusion_model'),
]
PROJECT_DIR = next((p for p in PROJECT_CANDIDATES if (p / 'scripts').exists()), Path.cwd()).resolve()
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

SEED = int(os.environ.get('CONDITIONAL_SEED', 123))
MAX_IMAGES_PER_GROUP = int(os.environ.get('CONDITIONAL_MAX_IMAGES_PER_GROUP', 6))
CLASS_SWEEP = 'nf_class_conditional_u128'
CONT_SWEEP = 'nf_conditional_u128'
CLASS_SAMPLE_LABEL = os.environ.get('CLASS_SAMPLE_LABEL', 'raw_class_conditional')
CONT_SAMPLE_LABEL = os.environ.get('CONT_SAMPLE_LABEL', 'raw_conditional')
OUT_DIR = PROJECT_DIR / 'results' / 'conditional_inference'
OUT_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 180,
    'font.size': 11,
})

print('project:', PROJECT_DIR)
print('seed:', SEED)
print('output:', OUT_DIR)

## Utilities

In [ ]:
def read_json(path: Path) -> Any:
    with path.open() as f:
        return json.load(f)


def manifest_row(sweep: str) -> dict[str, Any] | None:
    path = PROJECT_DIR / 'local' / sweep / 'manifest.json'
    if not path.exists():
        print('missing manifest:', path)
        return None
    rows = read_json(path)
    if not rows:
        print('empty manifest:', path)
        return None
    if isinstance(rows, dict):
        return rows
    if len(rows) > 1:
        print(f'{path}: expected one row, found {len(rows)}; using first row')
    return rows[0]


def resolve_project_path(raw: str | Path | None) -> Path | None:
    if raw is None:
        return None
    path = Path(str(raw))
    if path.is_absolute():
        if path.exists():
            return path
        # Some cached local manifests contain absolute paths from a different clone.
        marker = 'diffusion_models_repo'
        parts = path.parts
        if marker in parts:
            suffix = Path(*parts[parts.index(marker) + 1:])
            return PROJECT_DIR / suffix
        marker = 'Diffusion_model'
        if marker in parts:
            suffix = Path(*parts[parts.index(marker) + 1:])
            return PROJECT_DIR / suffix
        return path
    return PROJECT_DIR / path


def _format_candidate(raw: str | Path, row: dict[str, Any]) -> Path:
    return resolve_project_path(str(raw).format(seed=SEED, run_name=row.get('run_name', '')))


def resolve_sample_path(row: dict[str, Any], sweep: str, suffix_hint: str, sample_label: str | None = None) -> Path | None:
    run = row.get('run_name', '*')
    sample_root = PROJECT_DIR / 'results' / sweep / 'samples'
    candidates: list[Path] = []

    raw = row.get('sample_path')
    if raw:
        candidates.append(_format_candidate(raw, row))

    if sample_label:
        candidates.append(sample_root / f'{run}_seed{SEED}_{sample_label}.npz')
        candidates.extend(sorted(sample_root.glob(f'{run}_seed{SEED}_*{sample_label}*.npz')))

    candidates.extend(sorted(sample_root.glob(f'{run}_seed{SEED}_*.npz')))
    candidates.extend(sorted(sample_root.glob(f'*seed{SEED}*{suffix_hint}*.npz')))

    seen: set[Path] = set()
    unique_candidates = []
    for path in candidates:
        if path is None or path in seen:
            continue
        seen.add(path)
        unique_candidates.append(path)

    if sample_label:
        for path in unique_candidates:
            if path.exists() and sample_label in path.name:
                return path
    for path in unique_candidates:
        if path.exists():
            return path
    return unique_candidates[0] if unique_candidates else None


def load_npz_samples(path: Path) -> np.ndarray:
    with np.load(path) as data:
        key = 'samples' if 'samples' in data.files else data.files[0]
        arr = np.asarray(data[key], dtype=np.float32)
    if arr.ndim == 3:
        arr = arr[:, None, :, :]
    if arr.ndim != 4 or arr.shape[1] != 1:
        raise ValueError(f'Expected (N,1,H,W) or (N,H,W), got {arr.shape} from {path}')
    return arr


def sample_stats(images: np.ndarray, group: np.ndarray | None = None, group_name: str = 'group') -> pd.DataFrame:
    flat = images.reshape(len(images), -1)
    df = pd.DataFrame({
        'sample_mean': flat.mean(axis=1),
        'sample_std': flat.std(axis=1),
        'sample_p01': np.percentile(flat, 1, axis=1),
        'sample_p50': np.percentile(flat, 50, axis=1),
        'sample_p99': np.percentile(flat, 99, axis=1),
    })
    if group is not None:
        df[group_name] = group[:len(df)]
    return df


def finite_summary(images: np.ndarray) -> dict[str, float | int | tuple[int, ...]]:
    arr = np.asarray(images)
    finite = np.isfinite(arr)
    return {
        'shape': tuple(arr.shape),
        'finite': int(finite.sum()),
        'total': int(arr.size),
        'min': float(np.nanmin(arr)),
        'max': float(np.nanmax(arr)),
        'mean': float(np.nanmean(arr)),
        'std': float(np.nanstd(arr)),
    }


def robust_limits(*arrays: np.ndarray) -> tuple[float, float]:
    vals = np.concatenate([np.asarray(a).ravel() for a in arrays if np.asarray(a).size])
    lo, hi = np.nanpercentile(vals, [1, 99])
    if not np.isfinite(lo) or not np.isfinite(hi) or lo == hi:
        lo, hi = -1.0, 1.0
    return float(lo), float(hi)


def savefig(fig, name: str) -> Path:
    path = OUT_DIR / name
    fig.savefig(path, bbox_inches='tight')
    print('saved', path)
    return path


## Manifest And File Audit

In [ ]:
class_row = manifest_row(CLASS_SWEEP)
cont_row = manifest_row(CONT_SWEEP)

audit_rows = []
for sweep, row, hint, sample_label in [
    (CLASS_SWEEP, class_row, 'class', CLASS_SAMPLE_LABEL),
    (CONT_SWEEP, cont_row, 'conditional', CONT_SAMPLE_LABEL),
]:
    if row is None:
        audit_rows.append({'sweep': sweep, 'status': 'missing manifest'})
        continue
    sample_path = resolve_sample_path(row, sweep, hint, sample_label)
    config_path = resolve_project_path(row.get('config'))
    checkpoint_dir = resolve_project_path(row.get('checkpoint_dir'))
    audit_rows.append({
        'sweep': sweep,
        'run_name': row.get('run_name'),
        'conditioning': row.get('conditioning'),
        'sample_label': sample_label,
        'dataset_size': row.get('dataset_size'),
        'sample_path': str(sample_path) if sample_path else None,
        'sample_exists': bool(sample_path and sample_path.exists()),
        'sample_size_mb': round(sample_path.stat().st_size / 1024**2, 2) if sample_path and sample_path.exists() else np.nan,
        'config_exists': bool(config_path and config_path.exists()),
        'checkpoint_exists': bool(checkpoint_dir and checkpoint_dir.exists()),
    })

audit_df = pd.DataFrame(audit_rows)
display(audit_df)

# Discrete Class-Conditional Field Model

Class IDs are expected to map to CAMELS field types. The default requested mapping is `0=Mcdm`, `1=Mstar`, `2=HI`, `3=Mgas`, `4=Mtot`, `5=ne`.

In [ ]:
if class_row is None:
    class_samples = None
    print('No class-conditional manifest found.')
else:
    class_sample_path = resolve_sample_path(class_row, CLASS_SWEEP, 'class', CLASS_SAMPLE_LABEL)
    class_label_path = resolve_project_path(class_row.get('sample_label_path'))
    class_map_path = resolve_project_path(class_row.get('class_map_path'))
    class_map = class_row.get('class_map') or (read_json(class_map_path) if class_map_path and class_map_path.exists() else {})
    id_to_field = {int(v): str(k) for k, v in class_map.items()}
    print('sample:', class_sample_path)
    print('sample label:', CLASS_SAMPLE_LABEL)
    print('labels:', class_label_path)
    print('class_map:', id_to_field)
    if not class_sample_path or not class_sample_path.exists():
        class_samples = None
        print('Class-conditional sample file missing.')
    elif not class_label_path or not class_label_path.exists():
        class_samples = None
        print('Class-conditional sample label file missing.')
    else:
        class_samples = load_npz_samples(class_sample_path)
        class_labels = np.load(class_label_path).astype(int)
        n = min(len(class_samples), len(class_labels))
        class_samples = class_samples[:n]
        class_labels = class_labels[:n]
        class_names = np.array([id_to_field.get(int(x), f'class_{int(x)}') for x in class_labels])
        print('summary:', finite_summary(class_samples))
        display(pd.Series(class_names).value_counts().rename_axis('field').reset_index(name='n_samples'))

In [ ]:
if 'class_samples' in globals() and class_samples is not None:
    class_stats = sample_stats(class_samples, class_names, 'field')
    class_grouped = class_stats.groupby('field').agg(['mean', 'std', 'min', 'max']).round(5)
    display(class_grouped)
    class_stats_path = OUT_DIR / 'nf_class_conditional_u128_sample_stats.csv'
    class_stats.to_csv(class_stats_path, index=False)
    print('wrote', class_stats_path)
else:
    print('No class samples loaded; skipping class stats.')

In [ ]:
if 'class_samples' in globals() and class_samples is not None:
    fields = [id_to_field[i] for i in sorted(id_to_field)]
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
    bins = np.linspace(*robust_limits(class_samples), 180)
    for field in fields:
        arr = class_samples[class_names == field]
        if len(arr) == 0:
            continue
        axes[0].hist(arr.ravel(), bins=bins, density=True, histtype='step', lw=1.8, label=field)
    axes[0].set_title('class-conditional pixel distributions')
    axes[0].set_xlabel('normalized field value')
    axes[0].set_ylabel('density')
    axes[0].legend(frameon=False, fontsize=9)

    order = [field for field in fields if field in set(class_stats['field'])]
    data = [class_stats.loc[class_stats.field == field, 'sample_std'] for field in order]
    axes[1].boxplot(data, labels=order, showfliers=False)
    axes[1].set_title('per-sample std by requested class')
    axes[1].set_ylabel('sample std')
    axes[1].tick_params(axis='x', rotation=35)
    fig.tight_layout()
    savefig(fig, 'nf_class_conditional_u128_histograms.png')
    plt.show()
else:
    print('No class samples loaded; skipping class histograms.')

In [ ]:
if 'class_samples' in globals() and class_samples is not None:
    fields = [id_to_field[i] for i in sorted(id_to_field)]
    n_cols = min(MAX_IMAGES_PER_GROUP, 6)
    fig, axes = plt.subplots(len(fields), n_cols, figsize=(2.1 * n_cols, 2.05 * len(fields)), squeeze=False)
    vmin, vmax = robust_limits(class_samples)
    for row_idx, field in enumerate(fields):
        idxs = np.where(class_names == field)[0]
        chosen = idxs[np.linspace(0, len(idxs) - 1, n_cols, dtype=int)] if len(idxs) else []
        for col_idx in range(n_cols):
            ax = axes[row_idx, col_idx]
            if col_idx < len(chosen):
                i = int(chosen[col_idx])
                ax.imshow(class_samples[i, 0], origin='lower', cmap='viridis', vmin=vmin, vmax=vmax)
                ax.set_title(f'{field} #{i}', fontsize=9)
            ax.set_xticks([])
            ax.set_yticks([])
    fig.suptitle('Discrete class-conditional generated samples', y=1.01)
    fig.tight_layout()
    savefig(fig, 'nf_class_conditional_u128_image_grid.png')
    plt.show()
else:
    print('No class samples loaded; skipping class image grid.')

# Continuous Conditional Cosmology Model

This model conditions HI generation on the six CAMELS parameters: `Omega_m`, `sigma_8`, `A_SN1`, `A_AGN1`, `A_SN2`, and `A_AGN2`.

In [ ]:
if cont_row is None:
    cont_samples = None
    print('No continuous-conditional manifest found.')
else:
    cont_sample_path = resolve_sample_path(cont_row, CONT_SWEEP, 'conditional', CONT_SAMPLE_LABEL)
    cont_label_path = resolve_project_path(cont_row.get('sample_label_path'))
    cont_raw_path = resolve_project_path(cont_row.get('sample_raw_params_path'))
    stats_path = resolve_project_path(cont_row.get('param_stats_path'))
    param_names = list(cont_row.get('param_names') or [])
    print('sample:', cont_sample_path)
    print('sample label:', CONT_SAMPLE_LABEL)
    print('normalized labels:', cont_label_path)
    print('raw labels:', cont_raw_path)
    print('stats:', stats_path)
    if not cont_sample_path or not cont_sample_path.exists():
        cont_samples = None
        print('Continuous conditional sample file missing.')
    elif not cont_label_path or not cont_label_path.exists():
        cont_samples = None
        print('Continuous conditional label file missing.')
    else:
        cont_samples = load_npz_samples(cont_sample_path)
        cont_labels_norm = np.load(cont_label_path).astype(np.float32)
        if cont_raw_path and cont_raw_path.exists():
            cont_labels_raw = np.load(cont_raw_path).astype(np.float32)
        elif stats_path and stats_path.exists():
            stats = read_json(stats_path)
            mean = np.asarray(stats['mean'], dtype=np.float32)
            std = np.asarray(stats['std'], dtype=np.float32)
            cont_labels_raw = cont_labels_norm * std + mean
            if not param_names:
                param_names = list(stats.get('param_names', []))
        else:
            cont_labels_raw = cont_labels_norm.copy()
        if not param_names:
            param_names = [f'param_{i}' for i in range(cont_labels_norm.shape[1])]
        n = min(len(cont_samples), len(cont_labels_norm), len(cont_labels_raw))
        cont_samples = cont_samples[:n]
        cont_labels_norm = cont_labels_norm[:n]
        cont_labels_raw = cont_labels_raw[:n]
        print('summary:', finite_summary(cont_samples))
        param_df = pd.DataFrame(cont_labels_raw, columns=param_names)
        display(param_df.describe().T.round(5))

In [ ]:
if 'cont_samples' in globals() and cont_samples is not None:
    cont_stats = sample_stats(cont_samples)
    param_df = pd.DataFrame(cont_labels_raw, columns=param_names)
    cont_stats = pd.concat([cont_stats, param_df], axis=1)
    display(cont_stats[['sample_mean', 'sample_std', 'sample_p01', 'sample_p99', *param_names]].head())
    corr_cols = ['sample_mean', 'sample_std', 'sample_p01', 'sample_p99']
    corr = cont_stats[corr_cols + param_names].corr().loc[corr_cols, param_names]
    display(corr.round(3))
    cont_stats_path = OUT_DIR / 'nf_conditional_u128_sample_stats.csv'
    cont_stats.to_csv(cont_stats_path, index=False)
    print('wrote', cont_stats_path)
else:
    print('No continuous samples loaded; skipping continuous stats.')

In [ ]:
if 'cont_samples' in globals() and cont_samples is not None:
    fig, axes = plt.subplots(2, 3, figsize=(15, 8.2), squeeze=False)
    for ax, pname in zip(axes.ravel(), param_names):
        sc = ax.scatter(cont_stats[pname], cont_stats['sample_std'], c=cont_stats['sample_mean'], s=18, cmap='viridis')
        ax.set_xlabel(pname)
        ax.set_ylabel('generated sample std')
        ax.grid(alpha=0.25)
    fig.colorbar(sc, ax=axes.ravel().tolist(), label='generated sample mean', shrink=0.85)
    fig.suptitle('Continuous conditional samples: generated summary vs conditioning parameters', y=1.02)
    savefig(fig, 'nf_conditional_u128_param_scatter.png')
    plt.show()
else:
    print('No continuous samples loaded; skipping parameter scatter plot.')

In [ ]:
if 'cont_samples' in globals() and cont_samples is not None:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
    bins = np.linspace(*robust_limits(cont_samples), 180)
    axes[0].hist(cont_samples.ravel(), bins=bins, density=True, histtype='step', lw=2, color='tab:blue')
    axes[0].set_title('continuous conditional pixel distribution')
    axes[0].set_xlabel('normalized field value')
    axes[0].set_ylabel('density')

    axes[1].hist(cont_stats['sample_mean'], bins=50, alpha=0.65, label='sample mean')
    axes[1].hist(cont_stats['sample_std'], bins=50, alpha=0.65, label='sample std')
    axes[1].set_title('per-sample summaries')
    axes[1].legend(frameon=False)
    fig.tight_layout()
    savefig(fig, 'nf_conditional_u128_histograms.png')
    plt.show()
else:
    print('No continuous samples loaded; skipping continuous histograms.')

In [ ]:
if 'cont_samples' in globals() and cont_samples is not None:
    n_params = len(param_names)
    fig, axes = plt.subplots(n_params, 2, figsize=(5.0, 2.1 * n_params), squeeze=False)
    vmin, vmax = robust_limits(cont_samples)
    for row_idx, pname in enumerate(param_names):
        vals = cont_stats[pname].to_numpy()
        for col_idx, which in enumerate(['min', 'max']):
            idx = int(np.argmin(vals) if which == 'min' else np.argmax(vals))
            ax = axes[row_idx, col_idx]
            ax.imshow(cont_samples[idx, 0], origin='lower', cmap='viridis', vmin=vmin, vmax=vmax)
            ax.set_title(f'{pname} {which}\nidx={idx}, value={vals[idx]:.3g}', fontsize=9)
            ax.set_xticks([])
            ax.set_yticks([])
    fig.suptitle('Continuous conditional generated samples at parameter extremes', y=1.01)
    fig.tight_layout()
    savefig(fig, 'nf_conditional_u128_parameter_extremes.png')
    plt.show()
else:
    print('No continuous samples loaded; skipping parameter-extreme image grid.')

## Great Lakes Sampling Status Commands

If either sample is missing in the audit table, check the jobs and logs directly on Great Lakes. The continuous cosmology sampler needs the concrete `UNet2DConditionModel` checkpoint-load patch, so pull the latest branch before resubmitting.

```bash
cd /home/jiamingp/diffusion_models_repo
git pull --ff-only
sacct -j 51018045,51018033 --format=JobID,JobName%24,State,ExitCode,Elapsed,MaxRSS
ls -lh results/nf_class_conditional_u128/samples/*.npz results/nf_conditional_u128/samples/*.npz
```


## Output Files

In [ ]:
for path in sorted(OUT_DIR.glob('*')):
    if path.is_file():
        print(path, path.stat().st_size)